In [ ]:
import gluestick as gs
import pandas as pd
import os
import json
from utils.auth import OptiplyAuthenticator
from utils.tools import (snapshot_records, get_snapshot, delete_from_snapshot, concat_columns,
                          handle_invalid_dates, round_to_2, round_to_0, validate_attribute,
                          convert_to_bool, round_numeric_to_2, round_numeric_to_0)
import datetime
from ast import literal_eval
from utils.payloads import (get_product_payload, get_supplier_payload,
                            get_supplier_product_payload,
                            get_sell_order_withlines_payload,
                            get_buy_order_payload, get_buy_order_line_payload,
                            get_receipt_line_payload)
from utils.actions import post_optiply, patch_optiply, delete_optiply, get_optiply
import numpy as np
import gc

In [ ]:
# Standard directory structure for hotglue
ROOT_DIR = os.environ.get("ROOT_DIR", ".")
INPUT_DIR = f"{ROOT_DIR}/sync-output"
OUTPUT_DIR = f"{ROOT_DIR}/etl-output"
SNAPSHOT_DIR = f"{ROOT_DIR}/snapshots"
CONFIG_DIR = f"{ROOT_DIR}/config"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SNAPSHOT_DIR, exist_ok=True)

input_data = gs.reader(INPUT_DIR)
print("Available streams:", list(input_data.keys()))

In [ ]:
# ── Tenant config ─────────────────────────────────────────────────────────────
config_path = f"{SNAPSHOT_DIR}/tenant-config.json"

if os.path.exists(config_path):
    with open(config_path) as f:
        config = json.load(f)
else:
    raise Exception("No 'tenant-config.json' provided.")

api_creds = config["apiCredentials"]
auth = OptiplyAuthenticator(api_creds, config_path, OUTPUT_DIR)

##### Check Config Flags

In [ ]:
# ── Integration config ────────────────────────────────────────────────────────
job_config_path = f"./config.json"

if os.path.exists(job_config_path):
    with open(job_config_path) as f:
        job_config = json.load(f)
else:
    raise Exception("No 'config.json' provided.")

print("Config:", job_config)

sync_products = job_config.get("sync_products", True)
sync_suppliers = job_config.get("sync_suppliers", True)
sync_supplier_products = job_config.get("sync_supplier_products", True)
sync_sell_orders = job_config.get("sync_sell_orders", True)
sync_buy_orders = job_config.get("sync_buy_orders", True)

# Extend-specific: warehouse codes to include for stock aggregation.
# Configured in connector-config / config.json; exclude RMA/returns warehouses.
raw_allowed_warehouses = job_config.get("warehouse_codes", []) or []
if isinstance(raw_allowed_warehouses, str):
    allowed_warehouses = [code.strip() for code in raw_allowed_warehouses.split(",") if code.strip()]
elif isinstance(raw_allowed_warehouses, list):
    allowed_warehouses = [str(code).strip() for code in raw_allowed_warehouses if str(code).strip()]
else:
    allowed_warehouses = []

SELL_ORDER_COMPLETED_STATUSES = {
    "Delivered", "Invoiced", "Completed", "FullyDelivered", "Terminated"
}
CANCELLED_STATUSES = {"Cancelled", "Annulled"}
RECEIVED_ROW_STATUSES = {"received"}

print(f"Sync: products={sync_products}, suppliers={sync_suppliers}, "
      f"supplierProducts={sync_supplier_products}, sellOrders={sync_sell_orders}, "
      f"buyOrders={sync_buy_orders}")
print(f"Allowed warehouses: {allowed_warehouses}")

# ── Job-state flags ───────────────────────────────────────────────────────────
def read_json_file(filename):
    with open(filename, "r") as f:
        return json.loads(f.read())

job_state_path = os.path.join(ROOT_DIR, "state.json")
if os.path.exists(job_state_path):
    job_state = read_json_file(job_state_path)
    force_patch_products = job_state.get("force_patch_products", False)
else:
    force_patch_products = False

print(f"force_patch_products: {force_patch_products}")

# ── API creds shorthand ──────────────────────────────────────────────────────
api_creds_short = {
    "account_id": api_creds.get("account_id"),
    "couplingId": api_creds.get("couplingId"),
}

# Summary counters
summary = {
    'products': {'deleted': 0, 'created': 0, 'updated': 0},
    'suppliers': {'deleted': 0, 'created': 0, 'updated': 0},
    'supplierProducts': {'deleted': 0, 'created': 0, 'updated': 0},
    'sellOrders': {'deleted': 0, 'created': 0, 'updated': 0},
    'buyOrders': {'deleted': 0, 'created': 0, 'updated': 0},
    'buyOrderLines': {'deleted': 0, 'created': 0, 'updated': 0},
    'receiptLines': {'deleted': 0, 'created': 0, 'updated': 0},
}

In [ ]:
def get_payload_function(row, entity):
    """Dispatch to the correct payload builder for the given entity."""
    if entity == "products":
        return get_product_payload(row)
    elif entity == "suppliers":
        return get_supplier_payload(row)
    elif entity == "supplierProducts":
        return get_supplier_product_payload(row)
    elif entity == "sellOrders":
        return get_sell_order_withlines_payload(row)
    elif entity == "buyOrders":
        return get_buy_order_payload(row)
    elif entity == "buyOrderLines":
        return get_buy_order_line_payload(row)
    elif entity == "receiptLines":
        return get_receipt_line_payload(row)
    else:
        raise ValueError(f"Unknown entity: {entity}")

## Auxiliar Functions

In [ ]:
def check_sync_output(stream_name, alternatives=None):
    """Check if a stream exists in sync output, handling ._resource fork files."""
    names = [stream_name]
    if alternatives:
        names.extend(alternatives)
    for name in names:
        for key in input_data.keys():
            if key.startswith("._"):
                continue
            if name.lower() in key.lower():
                return input_data[key]
    return None


def clean_underscore_columns(df):
    """Remove columns starting/ending with underscore."""
    cols = [c for c in df.columns if not c.startswith('_') and not c.endswith('_')]
    return df[cols]


# --- Extend-specific helpers ---

def parse_nested_json(val):
    """Parse a nested JSON string or literal_eval string into a list of dicts."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            parsed = json.loads(val)
            return parsed if isinstance(parsed, list) else [parsed]
        except (json.JSONDecodeError, TypeError):
            try:
                parsed = literal_eval(val)
                return parsed if isinstance(parsed, list) else [parsed]
            except (ValueError, SyntaxError):
                return []
    return []


def get_extend_status_completed(status, is_open, is_received, rows):
    """Determine the completed datetime for an Extend PO.

    Returns a datetime string when the PO is fully received and closed,
    or None if still open.
    """
    status_str = str(status).strip() if pd.notna(status) else ""

    if status_str == "Received" and not is_open:
        parsed_rows = parse_nested_json(rows)
        max_date = None
        for row in parsed_rows:
            if isinstance(row, dict):
                scd = row.get("statusChangeDate")
                if scd and pd.notna(scd):
                    dt = handle_invalid_dates(scd)
                    if not pd.isna(dt):
                        if max_date is None or str(dt) > str(max_date):
                            max_date = dt
        return str(max_date) if max_date else None

    return None


def get_deleted_at(status):
    """Return current datetime if status is a cancelled state, else None."""
    status_str = str(status).strip() if pd.notna(status) else ""
    if status_str in CANCELLED_STATUSES:
        return datetime.datetime.utcnow().isoformat()
    return None


def warehouse_is_allowed(warehouse):
    warehouse_str = str(warehouse or "").strip()
    return not allowed_warehouses or warehouse_str in allowed_warehouses


def compute_stock_from_entries(entries, balance_key):
    total = 0
    for entry in entries:
        if not isinstance(entry, dict):
            continue
        warehouse = entry.get('warehouse', '')
        if not warehouse_is_allowed(warehouse):
            continue
        balance = pd.to_numeric(entry.get(balance_key), errors='coerce')
        if pd.notna(balance):
            total += float(balance)
    return round_to_0(total)


def build_product_availability_stock_levels(product_availability_raw):
    if product_availability_raw is None or len(product_availability_raw) == 0:
        return None

    availability = clean_underscore_columns(product_availability_raw.copy())
    if 'productNumber' not in availability.columns:
        return None

    if 'warehouse' in availability.columns:
        availability = availability[availability['warehouse'].apply(warehouse_is_allowed)]

    if 'physicalBalance' in availability.columns:
        availability['physicalBalance'] = pd.to_numeric(availability['physicalBalance'], errors='coerce').fillna(0)
    else:
        availability['physicalBalance'] = 0

    stock_levels = availability.groupby('productNumber', dropna=False)['physicalBalance'].sum().reset_index()
    stock_levels['remoteId'] = stock_levels['productNumber'].astype(str)
    stock_levels['stockLevel'] = stock_levels['physicalBalance'].apply(round_to_0)
    return stock_levels[['remoteId', 'stockLevel']]


def build_snapshot_backed_products(products_snapshot, stock_levels, known_remote_ids=None):
    if products_snapshot is None or stock_levels is None:
        return None

    known_remote_ids = set() if known_remote_ids is None else set(known_remote_ids)
    stock_only = stock_levels[~stock_levels['remoteId'].isin(known_remote_ids)].copy()
    if len(stock_only) == 0:
        return None

    business_columns = [
        'remoteId', 'name', 'eanCode', 'articleCode', 'price', 'status',
        'stockLevel', 'unlimitedStock', 'assembled', 'notBeingBought',
        'minimumStock', 'created_at', 'updated_at', 'deleted_at'
    ]
    business_columns = [col for col in business_columns if col in products_snapshot.columns]

    supplemental = products_snapshot[products_snapshot['remoteId'].isin(stock_only['remoteId'])][business_columns].copy()
    if len(supplemental) == 0:
        return None

    supplemental = supplemental.merge(stock_only, on='remoteId', how='inner', suffixes=('', '_availability'))
    supplemental['stockLevel'] = supplemental['stockLevel_availability']
    supplemental = supplemental.drop(columns=['stockLevel_availability'])
    return supplemental

## Products
### Custom Mapping
Extend Commerce Products -> Optiply Products

Products come from the `products` stream. Stock prefers the
`product_availability` stream (`physicalBalance`) and falls back to the
`warehouse_stock` aggregate when availability data is absent.

In [ ]:
# Products: Custom Mapping (Extend-specific)
products = None
products_raw = check_sync_output('products', ['Products'])
product_availability_raw = check_sync_output('product_availability', ['ProductAvailability'])
product_availability_stock = build_product_availability_stock_levels(product_availability_raw)
products_snapshot = get_snapshot("products", SNAPSHOT_DIR)
if products_snapshot is not None and len(products_snapshot) == 0:
    products_snapshot = None

if products_raw is not None and len(products_raw) > 0 and sync_products:
    products_raw = clean_underscore_columns(products_raw)

    products = pd.DataFrame()
    products['remoteId'] = products_raw['productNumber'].astype(str)
    products['name'] = products_raw['productName'].astype(str)

    # EAN code
    if 'gtinNumberList' in products_raw.columns:
        products['eanCode'] = products_raw['gtinNumberList'].apply(
            lambda x: str(x) if pd.notna(x) and str(x).strip() else None
        )
    else:
        products['eanCode'] = None

    # Article code (manufacturer product number)
    if 'manufacturerProductNumber' in products_raw.columns:
        products['articleCode'] = products_raw['manufacturerProductNumber'].apply(
            lambda x: str(x) if pd.notna(x) and str(x).strip() else None
        )
    else:
        products['articleCode'] = None

    # Price is synced by a separate workflow. Preserve the snapshot value on updates.
    products['price'] = None

    # Status from enabled flag
    if 'enabled' in products_raw.columns:
        products['status'] = products_raw['enabled'].apply(
            lambda x: 'enabled' if str(x).lower() in ('true', '1', 'yes') else 'disabled'
        )
    else:
        products['status'] = 'enabled'

    # Stock level fallback: aggregate availableBalance from products when the
    # dedicated ProductAvailability stream is not present for a product.
    def compute_stock(warehouse_stock_json):
        entries = parse_nested_json(warehouse_stock_json)
        return compute_stock_from_entries(entries, 'availableBalance')

    if 'warehouse_stock' in products_raw.columns:
        products['stockLevel'] = products_raw['warehouse_stock'].apply(compute_stock)
    else:
        products['stockLevel'] = np.nan

    products['unlimitedStock'] = False

    # Dates
    if 'createDate' in products_raw.columns:
        products['created_at'] = products_raw['createDate']
        products['updated_at'] = products_raw['createDate']
    else:
        products['created_at'] = None
        products['updated_at'] = None

    products['deleted_at'] = None
    products = products.dropna(subset=['name'])

    if products_snapshot is not None:
        snapshot_merge = products_snapshot[['remoteId', 'price', 'stockLevel']].copy()
        if 'price' in snapshot_merge.columns:
            snapshot_merge['price'] = pd.to_numeric(snapshot_merge['price'], errors='coerce')
        if 'stockLevel' in snapshot_merge.columns:
            snapshot_merge['stockLevel'] = pd.to_numeric(snapshot_merge['stockLevel'], errors='coerce')
        products = products.merge(snapshot_merge, on='remoteId', how='left', suffixes=('', '_snapshot'))
        if 'price_snapshot' in products.columns:
            products['price'] = products['price'].combine_first(products['price_snapshot'])
            products = products.drop(columns=['price_snapshot'])
        if 'stockLevel_snapshot' in products.columns:
            products['stockLevel'] = products['stockLevel'].combine_first(products['stockLevel_snapshot'])
            products = products.drop(columns=['stockLevel_snapshot'])

    if product_availability_stock is not None:
        products = products.merge(product_availability_stock, on='remoteId', how='left', suffixes=('', '_availability'))
        products['stockLevel'] = products['stockLevel_availability'].combine_first(products['stockLevel'])
        products = products.drop(columns=['stockLevel_availability'])

    stock_only_products = build_snapshot_backed_products(
        products_snapshot,
        product_availability_stock,
        known_remote_ids=products['remoteId'].tolist(),
    )
    if stock_only_products is not None:
        products = pd.concat([products, stock_only_products], ignore_index=True, sort=False)

    print(f"Products custom mapped: {len(products)}")
elif product_availability_stock is not None and products_snapshot is not None and sync_products:
    products = build_snapshot_backed_products(products_snapshot, product_availability_stock)
    if products is not None:
        print(f"Products custom mapped from stock-only availability updates: {len(products)}")
    else:
        print("No stock-only product updates resolved from snapshot.")
elif not sync_products:
    print("Products sync disabled by config.")
else:
    print("No products data found.")

### Global Mapping

In [ ]:
# Products: Global Mapping
print("Global Mapping Products")
if products is not None:
    products = clean_underscore_columns(products)
    products.columns = products.columns.str.lower()

    if "remoteid" in products.columns:
        products.rename(columns={"remoteid": "remoteId"}, inplace=True)
    products["remoteId"] = products["remoteId"].astype(str)

    if "name" in products.columns:
        products["name"] = products["name"].astype(str).str.replace("\r", "").str[:255]

    if "eancode" in products.columns:
        products.rename(columns={"eancode": "eanCode"}, inplace=True)
        products["eanCode"] = products["eanCode"].apply(lambda x: str(x)[:50] if pd.notna(x) else None)

    if "articlecode" in products.columns:
        products.rename(columns={"articlecode": "articleCode"}, inplace=True)
        products["articleCode"] = products["articleCode"].apply(lambda x: str(x)[:100] if pd.notna(x) else None)

    if "price" in products.columns:
        products["price"] = products["price"].apply(round_numeric_to_2)
        products["price"] = products["price"].clip(lower=0)

    if "stocklevel" in products.columns:
        products.rename(columns={"stocklevel": "stockLevel"}, inplace=True)
        products["stockLevel"] = products["stockLevel"].apply(round_to_0)

    if "unlimitedstock" in products.columns:
        products.rename(columns={"unlimitedstock": "unlimitedStock"}, inplace=True)

    # Date handling
    for date_col in ["created_at", "updated_at", "deleted_at"]:
        if date_col not in products.columns:
            products[date_col] = None
        elif products[date_col].dtype == 'object':
            products[date_col] = products[date_col].astype(str).str[:19] + "Z"
        else:
            products[date_col] = None

    # Clean up: remove null/empty/nan remoteIds, deduplicate
    products = products[products["remoteId"].notna()]
    products = products[products["remoteId"] != ""]
    products = products[products["remoteId"] != "nan"]
    products = products.drop_duplicates(subset=["remoteId"])

    # concat_attributes for change detection (exclude date columns)
    concat_fields = products.columns[~products.columns.isin(["created_at", "updated_at", "deleted_at"])]
    products["concat_attributes"] = concat_columns(products, concat_fields)

    if len(products) == 0:
        products = None
    else:
        print(f"Products after global mapping: {len(products)}")

In [ ]:
# Products: Snapshot comparison
if products_snapshot is not None and products is not None:
    products_snapshot["remoteId"] = products_snapshot["remoteId"].astype(str)
    products["remoteId"] = products["remoteId"].astype(str)

    new_products = products[~products["remoteId"].isin(products_snapshot["remoteId"])]
    update_products = products[products["remoteId"].isin(products_snapshot["remoteId"])]

    if len(new_products) == 0: new_products = None
    if len(update_products) == 0: update_products = None
else:
    new_products = products
    update_products = None

del products

# NEW: filter out deleted/disabled
if new_products is not None:
    new_products = new_products[new_products["deleted_at"].isnull() | (new_products["deleted_at"] == "")]
    if "status" in new_products.columns:
        new_products = new_products[new_products["status"] != "disabled"]
    if len(new_products) == 0: new_products = None

# UPDATE: merge with snapshot to get optiply_id + detect changes
if update_products is not None:
    update_products = update_products.merge(
        products_snapshot[["remoteId", "optiply_id", "concat_attributes"]],
        on="remoteId", how="left", suffixes=("", "_snap"))

    # Extract deletes from updates (deleted_at is set)
    delete_products = update_products[~update_products["deleted_at"].isnull() & (update_products["deleted_at"] != "")]
    delete_products = delete_products.drop(columns=["concat_attributes_snap"])

    # Keep non-deleted, changed records only
    update_products = update_products[update_products["deleted_at"].isnull() | (update_products["deleted_at"] == "")]
    update_products = update_products[update_products["concat_attributes"] != update_products["concat_attributes_snap"]]
    update_products = update_products.drop(columns=["concat_attributes_snap"])

    if len(update_products) == 0: update_products = None
    if len(delete_products) == 0: delete_products = None
else:
    delete_products = None

del products_snapshot

In [ ]:
# Set Information to use on final requests
new_records = new_products
del new_products

update_records = update_products
del update_products

delete_records = delete_products
del delete_products

entity = "products"
snapshot_name = "products"

print(f"Products -- DELETE: {len(delete_records) if delete_records is not None else 0}, "
      f"POST: {len(new_records) if new_records is not None else 0}, "
      f"PATCH: {len(update_records) if update_records is not None else 0}")

In [ ]:
# DELETE
if delete_records is not None:
    delete_records["optiply_id"] = delete_records["optiply_id"].astype(float).astype(int)
    total_records = len(delete_records)
    delete_records = delete_records.reset_index(drop=True)
    try:
        for i, row in delete_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            response = delete_optiply(api_creds_short, auth, optiply_id, entity=entity)
            if response.status_code == 404:
                print(f"Record {optiply_id} is already deleted in Optiply, skipping.")
                continue
            if not response.ok:
                raise Exception(f"Failed to delete record {optiply_id} - Status: {response.status_code}")
            summary[entity]['deleted'] += 1
        delete_from_snapshot(delete_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    except Exception as e:
        raise Exception(f"ETL FAILED WHILE DELETING RECORDS\n{e}")
    del delete_records

# POST
if new_records is not None:
    new_records["remoteId"] = new_records["remoteId"].astype(str)
    new_records_ = new_records.copy().reset_index(drop=True)
    new_records_["optiply_id"] = None
    total_records = len(new_records_)
    try:
        for i, row in new_records_.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            payload = get_payload_function(row, entity)
            response = post_optiply(api_creds_short, auth, payload, entity=entity)
            new_records_.loc[i, "optiply_id"] = str(response.json()["data"]["id"])
            summary[entity]['created'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE POSTING RECORDS -- SNAPSHOTTING POSTED RECORDS")
    finally:
        new_records_ = new_records_[~new_records_["optiply_id"].isna()]
        new_records_["optiply_id"] = new_records_["optiply_id"].astype(float).astype(int)
        snapshot_records(new_records_, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del new_records
    del new_records_

# PATCH
if update_records is not None:
    update_records["remoteId"] = update_records["remoteId"].astype(str)
    update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
    update_records["response_code"] = None
    total_records = len(update_records)
    update_records = update_records.reset_index(drop=True)
    try:
        for i, row in update_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            payload = get_payload_function(row, entity)
            response = patch_optiply(api_creds_short, auth, payload, optiply_id, entity=entity)
            update_records.loc[i, "response_code"] = response.status_code
            summary[entity]['updated'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE PATCHING RECORDS -- SNAPSHOTTING PATCHED RECORDS")
    finally:
        update_records = update_records[update_records["response_code"] == 200]
        update_records = update_records.drop(columns=["response_code"])
        update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
        snapshot_records(update_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del update_records

gc.collect()

## Suppliers
### Custom Mapping
Extend Commerce SupplierAgreements -> Optiply Suppliers

SupplierAgreements (contract/brand level) map to Optiply Suppliers.
Lead times are computed from production + transport hours -> days.

In [ ]:
# Suppliers: Custom Mapping (Extend-specific)
suppliers = None
sa_raw = check_sync_output('supplier_agreements', ['SupplierAgreements', 'supplierAgreements'])

if sa_raw is not None and len(sa_raw) > 0 and sync_suppliers:
    sa_raw = sa_raw.copy()

    def compute_lead_days(row):
        manufacturing_h = float(row.get('manufacturingLeadTime') or 0)
        transport_h = float(row.get('transportLeadTime') or 0)
        customs_h = float(row.get('customsLeadTime') or 0)
        total_h = manufacturing_h + transport_h + customs_h
        if total_h > 0:
            return round(total_h / 24)
        return None

    suppliers = pd.DataFrame()
    suppliers['remoteId'] = sa_raw['supplierAgreementNumber'].apply(
        lambda x: str(int(x)) if pd.notna(x) else None
    )
    suppliers['name'] = sa_raw['name'].astype(str)
    suppliers['deliveryTime'] = sa_raw.apply(compute_lead_days, axis=1)
    suppliers['updated_at'] = datetime.datetime.utcnow().isoformat()
    suppliers['created_at'] = None
    suppliers['deleted_at'] = None

    suppliers = suppliers.dropna(subset=['remoteId'])

    print(f"Suppliers custom mapped: {len(suppliers)}")
elif not sync_suppliers:
    print("Suppliers sync disabled by config.")
else:
    print("No supplier_agreements data found.")

### Global Mapping

In [ ]:
# Suppliers: Global Mapping
print("Global Mapping Suppliers")
if suppliers is not None:
    suppliers = clean_underscore_columns(suppliers)
    suppliers.columns = suppliers.columns.str.lower()

    if "remoteid" in suppliers.columns:
        suppliers.rename(columns={"remoteid": "remoteId"}, inplace=True)
    suppliers["remoteId"] = suppliers["remoteId"].astype(str)

    if "name" in suppliers.columns:
        suppliers["name"] = suppliers["name"].astype(str).str.replace("\r", "").str[:255]

    if "deliverytime" in suppliers.columns:
        suppliers.rename(columns={"deliverytime": "deliveryTime"}, inplace=True)
        suppliers["deliveryTime"] = pd.to_numeric(suppliers["deliveryTime"], errors="coerce")

    for date_col in ["created_at", "updated_at", "deleted_at"]:
        if date_col not in suppliers.columns:
            suppliers[date_col] = None
        elif suppliers[date_col].dtype == 'object':
            suppliers[date_col] = suppliers[date_col].astype(str).str[:19] + "Z"
        else:
            suppliers[date_col] = None

    suppliers = suppliers[suppliers["remoteId"].notna()]
    suppliers = suppliers[suppliers["remoteId"] != ""]
    suppliers = suppliers[suppliers["remoteId"] != "nan"]
    suppliers = suppliers.drop_duplicates(subset=["remoteId"])

    concat_fields = suppliers.columns[~suppliers.columns.isin(["created_at", "updated_at", "deleted_at"])]
    suppliers["concat_attributes"] = concat_columns(suppliers, concat_fields)

    if len(suppliers) == 0:
        suppliers = None
    else:
        print(f"Suppliers after global mapping: {len(suppliers)}")

In [ ]:
# Suppliers: Snapshot comparison
suppliers_snapshot = get_snapshot("suppliers", SNAPSHOT_DIR)
if suppliers_snapshot is not None and len(suppliers_snapshot) == 0:
    suppliers_snapshot = None

if suppliers_snapshot is not None and suppliers is not None:
    suppliers_snapshot["remoteId"] = suppliers_snapshot["remoteId"].astype(str)
    suppliers["remoteId"] = suppliers["remoteId"].astype(str)

    new_suppliers = suppliers[~suppliers["remoteId"].isin(suppliers_snapshot["remoteId"])]
    update_suppliers = suppliers[suppliers["remoteId"].isin(suppliers_snapshot["remoteId"])]

    if len(new_suppliers) == 0: new_suppliers = None
    if len(update_suppliers) == 0: update_suppliers = None
else:
    new_suppliers = suppliers
    update_suppliers = None

del suppliers

if new_suppliers is not None:
    new_suppliers = new_suppliers[new_suppliers["deleted_at"].isnull() | (new_suppliers["deleted_at"] == "")]
    if len(new_suppliers) == 0: new_suppliers = None

if update_suppliers is not None:
    update_suppliers = update_suppliers.merge(
        suppliers_snapshot[["remoteId", "optiply_id", "concat_attributes"]],
        on="remoteId", how="left", suffixes=("", "_snap"))

    delete_suppliers = update_suppliers[~update_suppliers["deleted_at"].isnull() & (update_suppliers["deleted_at"] != "")]
    delete_suppliers = delete_suppliers.drop(columns=["concat_attributes_snap"])

    update_suppliers = update_suppliers[update_suppliers["deleted_at"].isnull() | (update_suppliers["deleted_at"] == "")]
    update_suppliers = update_suppliers[update_suppliers["concat_attributes"] != update_suppliers["concat_attributes_snap"]]
    update_suppliers = update_suppliers.drop(columns=["concat_attributes_snap"])

    if len(update_suppliers) == 0: update_suppliers = None
    if len(delete_suppliers) == 0: delete_suppliers = None
else:
    delete_suppliers = None

del suppliers_snapshot

In [ ]:
new_records = new_suppliers
del new_suppliers
update_records = update_suppliers
del update_suppliers
delete_records = delete_suppliers
del delete_suppliers

entity = "suppliers"
snapshot_name = "suppliers"

print(f"Suppliers -- DELETE: {len(delete_records) if delete_records is not None else 0}, "
      f"POST: {len(new_records) if new_records is not None else 0}, "
      f"PATCH: {len(update_records) if update_records is not None else 0}")

In [ ]:
# DELETE
if delete_records is not None:
    delete_records["optiply_id"] = delete_records["optiply_id"].astype(float).astype(int)
    total_records = len(delete_records)
    delete_records = delete_records.reset_index(drop=True)
    try:
        for i, row in delete_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            response = delete_optiply(api_creds_short, auth, optiply_id, entity=entity)
            if response.status_code == 404:
                print(f"Record {optiply_id} is already deleted in Optiply, skipping.")
                continue
            if not response.ok:
                raise Exception(f"Failed to delete record {optiply_id} - Status: {response.status_code}")
            summary[entity]['deleted'] += 1
        delete_from_snapshot(delete_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    except Exception as e:
        raise Exception(f"ETL FAILED WHILE DELETING RECORDS\n{e}")
    del delete_records

# POST
if new_records is not None:
    new_records["remoteId"] = new_records["remoteId"].astype(str)
    new_records_ = new_records.copy().reset_index(drop=True)
    new_records_["optiply_id"] = None
    total_records = len(new_records_)
    try:
        for i, row in new_records_.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            payload = get_payload_function(row, entity)
            response = post_optiply(api_creds_short, auth, payload, entity=entity)
            new_records_.loc[i, "optiply_id"] = str(response.json()["data"]["id"])
            summary[entity]['created'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE POSTING RECORDS -- SNAPSHOTTING POSTED RECORDS")
    finally:
        new_records_ = new_records_[~new_records_["optiply_id"].isna()]
        new_records_["optiply_id"] = new_records_["optiply_id"].astype(float).astype(int)
        snapshot_records(new_records_, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del new_records
    del new_records_

# PATCH
if update_records is not None:
    update_records["remoteId"] = update_records["remoteId"].astype(str)
    update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
    update_records["response_code"] = None
    total_records = len(update_records)
    update_records = update_records.reset_index(drop=True)
    try:
        for i, row in update_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            payload = get_payload_function(row, entity)
            response = patch_optiply(api_creds_short, auth, payload, optiply_id, entity=entity)
            update_records.loc[i, "response_code"] = response.status_code
            summary[entity]['updated'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE PATCHING RECORDS -- SNAPSHOTTING PATCHED RECORDS")
    finally:
        update_records = update_records[update_records["response_code"] == 200]
        update_records = update_records.drop(columns=["response_code"])
        update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
        snapshot_records(update_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del update_records

gc.collect()

## Supplier Products
### Custom Mapping
Extend Commerce ProductSupplierAgreements -> Optiply SupplierProducts

One record per (product, supplier agreement) pair.
remoteId = `{productNumber}_{supplierAgreementNumber}`.
Inactive links are skipped. Price adjusted for VAT.

In [ ]:
# SupplierProducts: Custom Mapping (Extend-specific)
supplier_products = None
psa_raw = check_sync_output('product_supplier_agreements', ['ProductSupplierAgreements', 'productSupplierAgreements'])

if psa_raw is not None and len(psa_raw) > 0 and sync_supplier_products:
    products_snapshot = get_snapshot('products', SNAPSHOT_DIR)
    suppliers_snapshot = get_snapshot('suppliers', SNAPSHOT_DIR)

    psa = psa_raw.copy()

    # Skip inactive links
    if 'inactive' in psa.columns:
        psa = psa[~psa['inactive'].apply(lambda x: str(x).lower() == 'true')]

    # Build composite ID
    psa['agreement_str'] = psa['supplierAgreementNumber'].apply(
        lambda x: str(int(x)) if pd.notna(x) else None
    )
    psa = psa.dropna(subset=['productNumber', 'agreement_str'])
    psa['composite_id'] = psa['productNumber'].astype(str) + '_' + psa['agreement_str']

    # Resolve product and supplier Optiply IDs
    if products_snapshot is not None:
        prod_lookup = products_snapshot[["remoteId", "optiply_id"]].rename(
            columns={"remoteId": "productNumber", "optiply_id": "productId"})
        prod_lookup["productNumber"] = prod_lookup["productNumber"].astype(str)
        psa["productNumber"] = psa["productNumber"].astype(str)
        psa = psa.merge(prod_lookup, on="productNumber", how="left")
    else:
        psa["productId"] = None

    if suppliers_snapshot is not None:
        sup_lookup = suppliers_snapshot[["remoteId", "optiply_id"]].rename(
            columns={"remoteId": "agreement_str", "optiply_id": "supplierId"})
        sup_lookup["agreement_str"] = sup_lookup["agreement_str"].astype(str)
        psa = psa.merge(sup_lookup, on="agreement_str", how="left")
    else:
        psa["supplierId"] = None

    psa = psa.dropna(subset=['productId', 'supplierId'])

    psa['price_raw'] = pd.to_numeric(psa.get('price'), errors='coerce')

    def compute_supplier_product_delivery_days(row):
        direct_h = float(row.get('manufacturingLeadTimeHour') or 0)
        if direct_h > 0:
            return int(direct_h / 8)
        agreement_h = float(row.get('supplierAgreementProductionLeadtimeHours') or 0) + \
            float(row.get('supplierAgreementTransportLeadtimeHours') or 0)
        if agreement_h > 0:
            return int(agreement_h / 8)
        return None

    psa['supplier_product_name'] = psa['supplierProductName'].replace(r'^\\s*$', pd.NA, regex=True)
    psa['supplier_product_name'] = psa['supplier_product_name'].fillna(psa['supplierAgreementName'])

    psa['deliveryTime_days'] = psa.apply(compute_supplier_product_delivery_days, axis=1)

    supplier_products = pd.DataFrame({
        'remoteId':     psa['composite_id'].values,
        'name':         psa['supplier_product_name'].astype(str),
        'skuCode':      psa['supplierProductNumber'].apply(lambda x: str(x) if pd.notna(x) and str(x).strip() else None),
        'articleCode':  psa['productNumber'].astype(str).values,
        'price':        psa['price_raw'].values,
        'productId':    psa['productId'].astype(int).values,
        'supplierId':   psa['supplierId'].astype(int).values,
        'lotSize':      pd.to_numeric(psa.get('quantityPerPurchaseProductUnit'), errors='coerce').values,
        'deliveryTime': psa['deliveryTime_days'].values,
        'updated_at':   datetime.datetime.utcnow().isoformat(),
        'created_at':   None,
        'deleted_at':   None,
    })

    print(f"SupplierProducts custom mapped: {len(supplier_products)}")
elif not sync_supplier_products:
    print("SupplierProducts sync disabled by config.")
else:
    print("No product_supplier_agreements data found.")

### Global Mapping

In [ ]:
# SupplierProducts: Global Mapping
print("Global Mapping SupplierProducts")
if supplier_products is not None:
    supplier_products = clean_underscore_columns(supplier_products)
    supplier_products.columns = supplier_products.columns.str.lower()

    if "remoteid" in supplier_products.columns:
        supplier_products.rename(columns={"remoteid": "remoteId"}, inplace=True)
    supplier_products["remoteId"] = supplier_products["remoteId"].astype(str)

    if "name" in supplier_products.columns:
        supplier_products["name"] = supplier_products["name"].astype(str).str.replace("\r", "").str[:255]

    if "skucode" in supplier_products.columns:
        supplier_products.rename(columns={"skucode": "skuCode"}, inplace=True)
        supplier_products["skuCode"] = supplier_products["skuCode"].apply(lambda x: str(x)[:100] if pd.notna(x) else None)

    if "price" in supplier_products.columns:
        supplier_products["price"] = round_numeric_to_2(supplier_products["price"])
        supplier_products["price"] = supplier_products["price"].clip(lower=0)

    if "productid" in supplier_products.columns:
        supplier_products.rename(columns={"productid": "productId"}, inplace=True)
    if "supplierid" in supplier_products.columns:
        supplier_products.rename(columns={"supplierid": "supplierId"}, inplace=True)
    if "deliverytime" in supplier_products.columns:
        supplier_products.rename(columns={"deliverytime": "deliveryTime"}, inplace=True)
        supplier_products["deliveryTime"] = pd.to_numeric(supplier_products["deliveryTime"], errors="coerce")

    for date_col in ["created_at", "updated_at", "deleted_at"]:
        if date_col not in supplier_products.columns:
            supplier_products[date_col] = None
        elif supplier_products[date_col].dtype == 'object':
            supplier_products[date_col] = supplier_products[date_col].astype(str).str[:19] + "Z"
        else:
            supplier_products[date_col] = None

    supplier_products = supplier_products[supplier_products["remoteId"].notna()]
    supplier_products = supplier_products[supplier_products["remoteId"] != ""]
    supplier_products = supplier_products[supplier_products["remoteId"] != "nan"]
    supplier_products = supplier_products.drop_duplicates(subset=["remoteId"])

    concat_fields = supplier_products.columns[~supplier_products.columns.isin(
        ["created_at", "updated_at", "deleted_at", "productId", "supplierId"])]
    supplier_products["concat_attributes"] = concat_columns(supplier_products, concat_fields)

    if len(supplier_products) == 0:
        supplier_products = None
    else:
        print(f"SupplierProducts after global mapping: {len(supplier_products)}")

In [ ]:
# SupplierProducts: Snapshot comparison
sp_snapshot = get_snapshot("supplierProducts", SNAPSHOT_DIR)
if sp_snapshot is not None and len(sp_snapshot) == 0:
    sp_snapshot = None

if sp_snapshot is not None and supplier_products is not None:
    sp_snapshot["remoteId"] = sp_snapshot["remoteId"].astype(str)
    supplier_products["remoteId"] = supplier_products["remoteId"].astype(str)

    new_sp = supplier_products[~supplier_products["remoteId"].isin(sp_snapshot["remoteId"])]
    update_sp = supplier_products[supplier_products["remoteId"].isin(sp_snapshot["remoteId"])]

    if len(new_sp) == 0: new_sp = None
    if len(update_sp) == 0: update_sp = None
else:
    new_sp = supplier_products
    update_sp = None

del supplier_products

if new_sp is not None:
    new_sp = new_sp[new_sp["deleted_at"].isnull() | (new_sp["deleted_at"] == "")]
    if len(new_sp) == 0: new_sp = None

if update_sp is not None:
    update_sp = update_sp.merge(
        sp_snapshot[["remoteId", "optiply_id", "concat_attributes"]],
        on="remoteId", how="left", suffixes=("", "_snap"))

    delete_sp = update_sp[~update_sp["deleted_at"].isnull() & (update_sp["deleted_at"] != "")]
    delete_sp = delete_sp.drop(columns=["concat_attributes_snap"])

    update_sp = update_sp[update_sp["deleted_at"].isnull() | (update_sp["deleted_at"] == "")]
    update_sp = update_sp[update_sp["concat_attributes"] != update_sp["concat_attributes_snap"]]
    update_sp = update_sp.drop(columns=["concat_attributes_snap"])

    if len(update_sp) == 0: update_sp = None
    if len(delete_sp) == 0: delete_sp = None
else:
    delete_sp = None

del sp_snapshot

In [ ]:
new_records = new_sp
del new_sp
update_records = update_sp
del update_sp
delete_records = delete_sp
del delete_sp

entity = "supplierProducts"
snapshot_name = "supplierProducts"

print(f"SupplierProducts -- DELETE: {len(delete_records) if delete_records is not None else 0}, "
      f"POST: {len(new_records) if new_records is not None else 0}, "
      f"PATCH: {len(update_records) if update_records is not None else 0}")

In [ ]:
# DELETE
if delete_records is not None:
    delete_records["optiply_id"] = delete_records["optiply_id"].astype(float).astype(int)
    total_records = len(delete_records)
    delete_records = delete_records.reset_index(drop=True)
    try:
        for i, row in delete_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            response = delete_optiply(api_creds_short, auth, optiply_id, entity=entity)
            if response.status_code == 404:
                print(f"Record {optiply_id} is already deleted in Optiply, skipping.")
                continue
            if not response.ok:
                raise Exception(f"Failed to delete record {optiply_id} - Status: {response.status_code}")
            summary[entity]['deleted'] += 1
        delete_from_snapshot(delete_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    except Exception as e:
        raise Exception(f"ETL FAILED WHILE DELETING RECORDS\n{e}")
    del delete_records

# POST
if new_records is not None:
    new_records["remoteId"] = new_records["remoteId"].astype(str)
    new_records_ = new_records.copy().reset_index(drop=True)
    new_records_["optiply_id"] = None
    total_records = len(new_records_)
    try:
        for i, row in new_records_.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            payload = get_payload_function(row, entity)
            response = post_optiply(api_creds_short, auth, payload, entity=entity)
            new_records_.loc[i, "optiply_id"] = str(response.json()["data"]["id"])
            summary[entity]['created'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE POSTING RECORDS -- SNAPSHOTTING POSTED RECORDS")
    finally:
        new_records_ = new_records_[~new_records_["optiply_id"].isna()]
        new_records_["optiply_id"] = new_records_["optiply_id"].astype(float).astype(int)
        snapshot_records(new_records_, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del new_records
    del new_records_

# PATCH
if update_records is not None:
    update_records["remoteId"] = update_records["remoteId"].astype(str)
    update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
    update_records["response_code"] = None
    total_records = len(update_records)
    update_records = update_records.reset_index(drop=True)
    try:
        for i, row in update_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            payload = get_payload_function(row, entity)
            response = patch_optiply(api_creds_short, auth, payload, optiply_id, entity=entity)
            update_records.loc[i, "response_code"] = response.status_code
            summary[entity]['updated'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE PATCHING RECORDS -- SNAPSHOTTING PATCHED RECORDS")
    finally:
        update_records = update_records[update_records["response_code"] == 200]
        update_records = update_records.drop(columns=["response_code"])
        update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
        snapshot_records(update_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del update_records

gc.collect()

## Sell Orders
### Custom Mapping
Extend Commerce CustomerOrders -> Optiply SellOrders

CustomerOrders use `modifiedDateFrom`. Order lines are extracted from
`order_rows` JSON and embedded in the SellOrder payload via
`get_sell_order_withlines_payload`.

In [ ]:
# SellOrders: Custom Mapping (Extend-specific)
sell_orders = None
sell_order_lines = None
COMPLETED_STATUSES = SELL_ORDER_COMPLETED_STATUSES

so_raw = check_sync_output('customer_orders', ['customerOrders', 'CustomerOrders', 'sellOrders'])

if so_raw is not None and len(so_raw) > 0 and sync_sell_orders:
    so_raw = clean_underscore_columns(so_raw)

    sell_orders = pd.DataFrame()
    sell_orders['remoteId'] = so_raw['orderNumber'].astype(str)

    # Placed date
    if 'orderDate' in so_raw.columns:
        sell_orders['placed'] = so_raw['orderDate'].apply(
            lambda x: handle_invalid_dates(x) if pd.notna(x) else pd.NaT
        )
    else:
        sell_orders['placed'] = pd.NaT
    sell_orders = sell_orders.dropna(subset=['placed'])

    # Total value (already excl. VAT)
    if 'totalPrice' in so_raw.columns:
        sell_orders['totalValue'] = pd.to_numeric(
            so_raw.loc[sell_orders.index, 'totalPrice'], errors='coerce'
        ).fillna(0)
    else:
        sell_orders['totalValue'] = 0.0

    # Status handling
    status_col = so_raw.loc[sell_orders.index, 'orderStatus'].astype(str).str.strip() if 'orderStatus' in so_raw.columns else pd.Series('', index=sell_orders.index)
    change_dates = so_raw.loc[sell_orders.index, 'changeDate'].apply(
        lambda x: handle_invalid_dates(x) if pd.notna(x) else pd.NaT
    ) if 'changeDate' in so_raw.columns else pd.Series(pd.NaT, index=sell_orders.index)

    sell_orders['updated_at'] = change_dates
    sell_orders['created_at'] = sell_orders['placed']

    # Completed: set changeDate if status is a completed status
    sell_orders['completed'] = None
    completed_mask = status_col.isin(COMPLETED_STATUSES)
    sell_orders.loc[completed_mask, 'completed'] = change_dates[completed_mask]

    # Deleted: set changeDate if status is Cancelled
    sell_orders['deleted_at'] = None
    cancelled_mask = status_col.isin(CANCELLED_STATUSES)
    sell_orders.loc[cancelled_mask, 'deleted_at'] = change_dates[cancelled_mask]

    # Keep order_rows JSON for SellOrderLines extraction
    sell_orders['order_rows_json'] = so_raw.loc[sell_orders.index, 'order_rows'].values if 'order_rows' in so_raw.columns else '[]'

    # --- Extract SellOrderLines ---
    products_snapshot = get_snapshot('products', SNAPSHOT_DIR)
    sol_records = []
    for idx, row in sell_orders.iterrows():
        order_number = str(row.get('remoteId', ''))
        if not order_number or order_number == 'nan':
            continue
        order_rows = parse_nested_json(row.get('order_rows_json', '[]'))
        for line in order_rows:
            if not isinstance(line, dict):
                continue
            if line.get('supplyMode', '') == 'Freight':
                continue
            product_info = line.get('product', {}) or {}
            product_number = product_info.get('productNumber')
            if not product_number:
                continue
            position = line.get('position', 0)
            sales_data = line.get('salesData', {}) or {}
            qty = int(float(sales_data.get('quantity', 0) or 0))
            unit_price = float(sales_data.get('unitPrice', 0) or 0)
            subtotal = round(qty * unit_price, 2)

            product_optiply_id = None
            if products_snapshot is not None:
                match_p = products_snapshot[products_snapshot['remoteId'] == str(product_number)]
                if len(match_p) > 0 and pd.notna(match_p.iloc[0].get('optiply_id')):
                    product_optiply_id = match_p.iloc[0]['optiply_id']
            if product_optiply_id is None:
                continue

            sol_records.append({
                'remoteId': f"{order_number}-{position}",
                'quantity': qty,
                'productId': int(product_optiply_id),
                'sellOrderId': order_number,
                'subtotalValue': subtotal,
            })

    if sol_records:
        sell_order_lines = pd.DataFrame(sol_records).drop_duplicates(subset=['remoteId'])
        print(f"Sell order lines extracted: {len(sell_order_lines)}")

    # Drop helper column
    sell_orders = sell_orders.drop(columns=['order_rows_json'], errors='ignore')

    print(f"SellOrders custom mapped: {len(sell_orders)}")
elif not sync_sell_orders:
    print("Sell orders sync disabled by config.")
else:
    print("No customer orders data found.")

### Global Mapping

In [ ]:
# SellOrders: Global Mapping
print("Global Mapping SellOrders")
if sell_orders is not None:
    sell_orders = clean_underscore_columns(sell_orders)
    sell_orders.columns = sell_orders.columns.str.lower()

    if "remoteid" in sell_orders.columns:
        sell_orders.rename(columns={"remoteid": "remoteId"}, inplace=True)
    sell_orders["remoteId"] = sell_orders["remoteId"].astype(str)

    if "totalvalue" in sell_orders.columns:
        sell_orders.rename(columns={"totalvalue": "totalValue"}, inplace=True)
        sell_orders["totalValue"] = sell_orders["totalValue"].apply(round_numeric_to_2)

    for date_col in ["placed", "completed", "created_at", "updated_at", "deleted_at"]:
        if date_col not in sell_orders.columns:
            sell_orders[date_col] = None
        elif sell_orders[date_col].dtype == 'object':
            sell_orders[date_col] = sell_orders[date_col].astype(str).str[:19] + "Z"

    sell_orders = sell_orders[sell_orders["remoteId"].notna()]
    sell_orders = sell_orders[sell_orders["remoteId"] != ""]
    sell_orders = sell_orders[sell_orders["remoteId"] != "nan"]
    sell_orders = sell_orders.drop_duplicates(subset=["remoteId"])

    concat_fields = sell_orders.columns[~sell_orders.columns.isin(["created_at", "updated_at", "deleted_at"])]
    sell_orders["concat_attributes"] = concat_columns(sell_orders, concat_fields)

    if len(sell_orders) == 0:
        sell_orders = None
    else:
        print(f"SellOrders after global mapping: {len(sell_orders)}")

# Attach order lines to sell orders for embedded payload
if sell_orders is not None and sell_order_lines is not None and len(sell_order_lines) > 0:
    lines_grouped = sell_order_lines.groupby('sellOrderId').apply(
        lambda x: x[['quantity', 'subtotalValue', 'productId']].to_dict('records')
    )
    sell_orders = sell_orders.merge(
        lines_grouped.rename('orderLines'),
        left_on='remoteId', right_index=True, how='left'
    )
    sell_orders['orderLines'] = sell_orders['orderLines'].apply(
        lambda x: x if isinstance(x, list) else []
    )
else:
    if sell_orders is not None:
        sell_orders['orderLines'] = sell_orders['remoteId'].apply(lambda x: [])

In [ ]:
# SellOrders: Snapshot comparison
so_snapshot = get_snapshot("sellOrders", SNAPSHOT_DIR)
if so_snapshot is not None and len(so_snapshot) == 0:
    so_snapshot = None

if so_snapshot is not None and sell_orders is not None:
    so_snapshot["remoteId"] = so_snapshot["remoteId"].astype(str)
    sell_orders["remoteId"] = sell_orders["remoteId"].astype(str)

    new_sell_orders = sell_orders[~sell_orders["remoteId"].isin(so_snapshot["remoteId"])]
    update_sell_orders = sell_orders[sell_orders["remoteId"].isin(so_snapshot["remoteId"])]

    if len(new_sell_orders) == 0: new_sell_orders = None
    if len(update_sell_orders) == 0: update_sell_orders = None
else:
    new_sell_orders = sell_orders
    update_sell_orders = None

del sell_orders

if new_sell_orders is not None:
    new_sell_orders = new_sell_orders[new_sell_orders["deleted_at"].isnull() | (new_sell_orders["deleted_at"] == "")]
    if len(new_sell_orders) == 0: new_sell_orders = None

if update_sell_orders is not None:
    update_sell_orders = update_sell_orders.merge(
        so_snapshot[["remoteId", "optiply_id", "concat_attributes"]],
        on="remoteId", how="left", suffixes=("", "_snap"))

    delete_sell_orders = update_sell_orders[~update_sell_orders["deleted_at"].isnull() & (update_sell_orders["deleted_at"] != "")]
    delete_sell_orders = delete_sell_orders.drop(columns=["concat_attributes_snap"])

    update_sell_orders = update_sell_orders[update_sell_orders["deleted_at"].isnull() | (update_sell_orders["deleted_at"] == "")]
    update_sell_orders = update_sell_orders[update_sell_orders["concat_attributes"] != update_sell_orders["concat_attributes_snap"]]
    update_sell_orders = update_sell_orders.drop(columns=["concat_attributes_snap"])

    if len(update_sell_orders) == 0: update_sell_orders = None
    if len(delete_sell_orders) == 0: delete_sell_orders = None
else:
    delete_sell_orders = None

del so_snapshot

In [ ]:
new_records = new_sell_orders
del new_sell_orders
update_records = update_sell_orders
del update_sell_orders
delete_records = delete_sell_orders
del delete_sell_orders

entity = "sellOrders"
snapshot_name = "sellOrders"

print(f"SellOrders -- DELETE: {len(delete_records) if delete_records is not None else 0}, "
      f"POST: {len(new_records) if new_records is not None else 0}, "
      f"PATCH: {len(update_records) if update_records is not None else 0}")

In [ ]:
# DELETE
if delete_records is not None:
    delete_records["optiply_id"] = delete_records["optiply_id"].astype(float).astype(int)
    total_records = len(delete_records)
    delete_records = delete_records.reset_index(drop=True)
    try:
        for i, row in delete_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            response = delete_optiply(api_creds_short, auth, optiply_id, entity=entity)
            if response.status_code == 404:
                print(f"Record {optiply_id} is already deleted in Optiply, skipping.")
                continue
            if not response.ok:
                raise Exception(f"Failed to delete record {optiply_id} - Status: {response.status_code}")
            summary[entity]['deleted'] += 1
        delete_from_snapshot(delete_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    except Exception as e:
        raise Exception(f"ETL FAILED WHILE DELETING RECORDS\n{e}")
    del delete_records

# POST
if new_records is not None:
    new_records["remoteId"] = new_records["remoteId"].astype(str)
    new_records_ = new_records.copy().reset_index(drop=True)
    new_records_["optiply_id"] = None
    total_records = len(new_records_)
    try:
        for i, row in new_records_.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            payload = get_payload_function(row, entity)
            response = post_optiply(api_creds_short, auth, payload, entity=entity)
            new_records_.loc[i, "optiply_id"] = str(response.json()["data"]["id"])
            summary[entity]['created'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE POSTING RECORDS -- SNAPSHOTTING POSTED RECORDS")
    finally:
        new_records_ = new_records_[~new_records_["optiply_id"].isna()]
        new_records_["optiply_id"] = new_records_["optiply_id"].astype(float).astype(int)
        snapshot_records(new_records_, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del new_records
    del new_records_

# PATCH
if update_records is not None:
    update_records["remoteId"] = update_records["remoteId"].astype(str)
    update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
    update_records["response_code"] = None
    total_records = len(update_records)
    update_records = update_records.reset_index(drop=True)
    try:
        for i, row in update_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            payload = get_payload_function(row, entity)
            response = patch_optiply(api_creds_short, auth, payload, optiply_id, entity=entity)
            update_records.loc[i, "response_code"] = response.status_code
            summary[entity]['updated'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE PATCHING RECORDS -- SNAPSHOTTING PATCHED RECORDS")
    finally:
        update_records = update_records[update_records["response_code"] == 200]
        update_records = update_records.drop(columns=["response_code"])
        update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
        snapshot_records(update_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del update_records

gc.collect()

## Buy Orders
### Custom Mapping
Extend Commerce PurchaseOrders -> Optiply BuyOrders

**Bidirectional guard:** Orders where `externalOrderNumber` or `reference`
contains 'optiply' (case-insensitive) are filtered out.

In [ ]:
# BuyOrders: Custom Mapping (Extend-specific)
buy_orders = None
po_raw = check_sync_output('purchase_orders', ['purchaseOrders', 'PurchaseOrders', 'buyOrders'])

if po_raw is not None and len(po_raw) > 0 and sync_buy_orders:
    po_raw = clean_underscore_columns(po_raw)

    buy_orders = pd.DataFrame()
    buy_orders['remoteId'] = po_raw['purchaseNumber'].astype(str)

    # Placed date from createDate
    if 'createDate' in po_raw.columns:
        buy_orders['placed'] = po_raw['createDate'].apply(
            lambda x: handle_invalid_dates(x) if pd.notna(x) else pd.NaT
        )
    else:
        buy_orders['placed'] = pd.NaT
    buy_orders = buy_orders.dropna(subset=['placed'])

    # Supplier remote ID — use supplierAgreementNumber (FK to SupplierAgreement)
    if 'supplierAgreementNumber' in po_raw.columns:
        buy_orders['supplierRemoteId'] = po_raw.loc[buy_orders.index, 'supplierAgreementNumber'].apply(
            lambda x: str(int(x)) if pd.notna(x) else None
        )
    else:
        buy_orders['supplierRemoteId'] = None

    # Bidirectional guard fields
    buy_orders['externalOrderNumber'] = po_raw.loc[buy_orders.index, 'externalOrderNumber'].astype(str) if 'externalOrderNumber' in po_raw.columns else ''
    buy_orders['reference'] = po_raw.loc[buy_orders.index, 'reference'].astype(str) if 'reference' in po_raw.columns else ''

    # Status fields
    buy_orders['status'] = po_raw.loc[buy_orders.index, 'status'].astype(str) if 'status' in po_raw.columns else ''
    buy_orders['isOpen'] = po_raw.loc[buy_orders.index, 'isOpen'] if 'isOpen' in po_raw.columns else True
    buy_orders['rows_json'] = po_raw.loc[buy_orders.index, 'rows'] if 'rows' in po_raw.columns else '[]'

    # Total value: sum(qty * unitPrice) across all rows
    def compute_bo_total(row_idx):
        rows_val = po_raw.loc[row_idx].get('rows', '[]')
        parsed_rows = parse_nested_json(rows_val)
        total = 0.0
        for r in parsed_rows:
            if not isinstance(r, dict):
                continue
            product_unit = r.get('purchaseDataProductUnit', {}) or {}
            qty = float(product_unit.get('quantity', 0) or 0)
            unit_price = float(product_unit.get('unitPrice', 0) or 0)
            total += qty * unit_price
        return round(total, 2)

    buy_orders['totalValue'] = buy_orders.index.map(compute_bo_total)

    # Expected delivery date from PO header
    if 'requestedDeliveryDate' in po_raw.columns:
        buy_orders['expectedDeliveryDate'] = po_raw.loc[buy_orders.index, 'requestedDeliveryDate'].apply(
            lambda x: str(handle_invalid_dates(x)) if pd.notna(x) and not pd.isna(handle_invalid_dates(x)) else None
        )
    else:
        buy_orders['expectedDeliveryDate'] = None

    # Completed: Received AND !isOpen
    buy_orders['completed'] = buy_orders.apply(
        lambda row: get_extend_status_completed(
            row['status'], row['isOpen'], False, row['rows_json']
        ), axis=1
    )

    # Deleted at: Cancelled status
    buy_orders['deleted_at'] = buy_orders['status'].apply(get_deleted_at)
    buy_orders['updated_at'] = buy_orders['placed']
    buy_orders['created_at'] = buy_orders['placed']

    # --- Bidirectional guard ---
    pre_filter = len(buy_orders)
    optiply_mask = (
        buy_orders['externalOrderNumber'].str.lower().str.contains('optiply', na=False) |
        buy_orders['reference'].str.lower().str.contains('optiply', na=False)
    )
    buy_orders = buy_orders[~optiply_mask]
    filtered = pre_filter - len(buy_orders)
    if filtered > 0:
        print(f"Bidirectional guard: filtered out {filtered} Optiply-created buy orders")

    # Drop helper columns
    buy_orders = buy_orders.drop(columns=['externalOrderNumber', 'reference', 'status', 'isOpen', 'rows_json'], errors='ignore')

    # --- Resolve supplierId ---
    suppliers_snapshot = get_snapshot('suppliers', SNAPSHOT_DIR)
    if suppliers_snapshot is not None:
        sup_lookup = suppliers_snapshot[["remoteId", "optiply_id"]].rename(
            columns={"remoteId": "supplierRemoteId", "optiply_id": "supplierId"})
        sup_lookup["supplierRemoteId"] = sup_lookup["supplierRemoteId"].astype(str)
        buy_orders = buy_orders.merge(sup_lookup, on="supplierRemoteId", how="left")
    else:
        buy_orders["supplierId"] = None
    buy_orders = buy_orders[buy_orders['supplierId'].notna()]
    buy_orders = buy_orders.drop(columns=['supplierRemoteId'], errors='ignore')

    print(f"BuyOrders custom mapped: {len(buy_orders)}")
elif not sync_buy_orders:
    print("Buy orders sync disabled by config.")
else:
    print("No purchase orders data found.")

### Global Mapping

In [ ]:
# BuyOrders: Global Mapping
print("Global Mapping BuyOrders")
if buy_orders is not None:
    buy_orders = clean_underscore_columns(buy_orders)
    buy_orders.columns = buy_orders.columns.str.lower()

    if "remoteid" in buy_orders.columns:
        buy_orders.rename(columns={"remoteid": "remoteId"}, inplace=True)
    buy_orders["remoteId"] = buy_orders["remoteId"].astype(str)

    if "totalvalue" in buy_orders.columns:
        buy_orders.rename(columns={"totalvalue": "totalValue"}, inplace=True)
        buy_orders["totalValue"] = buy_orders["totalValue"].apply(round_numeric_to_2)

    if "supplierid" in buy_orders.columns:
        buy_orders.rename(columns={"supplierid": "supplierId"}, inplace=True)

    for date_col in ["placed", "completed", "expectedDeliveryDate", "created_at", "updated_at", "deleted_at"]:
        if date_col not in buy_orders.columns:
            buy_orders[date_col] = None
        elif buy_orders[date_col].dtype == 'object':
            buy_orders[date_col] = buy_orders[date_col].astype(str).str[:19] + "Z"

    buy_orders = buy_orders[buy_orders["remoteId"].notna()]
    buy_orders = buy_orders[buy_orders["remoteId"] != ""]
    buy_orders = buy_orders[buy_orders["remoteId"] != "nan"]
    buy_orders = buy_orders.drop_duplicates(subset=["remoteId"])

    concat_fields = buy_orders.columns[~buy_orders.columns.isin(["created_at", "updated_at", "deleted_at", "supplierId", "expectedDeliveryDate"])]
    buy_orders["concat_attributes"] = concat_columns(buy_orders, concat_fields)

    if len(buy_orders) == 0:
        buy_orders = None
    else:
        print(f"BuyOrders after global mapping: {len(buy_orders)}")

In [ ]:
# BuyOrders: Snapshot comparison
bo_snapshot = get_snapshot("buyOrders", SNAPSHOT_DIR)
if bo_snapshot is not None and len(bo_snapshot) == 0:
    bo_snapshot = None

if bo_snapshot is not None and buy_orders is not None:
    bo_snapshot["remoteId"] = bo_snapshot["remoteId"].astype(str)
    buy_orders["remoteId"] = buy_orders["remoteId"].astype(str)

    new_buy_orders = buy_orders[~buy_orders["remoteId"].isin(bo_snapshot["remoteId"])]
    update_buy_orders = buy_orders[buy_orders["remoteId"].isin(bo_snapshot["remoteId"])]

    if len(new_buy_orders) == 0: new_buy_orders = None
    if len(update_buy_orders) == 0: update_buy_orders = None
else:
    new_buy_orders = buy_orders
    update_buy_orders = None

del buy_orders

if new_buy_orders is not None:
    new_buy_orders = new_buy_orders[new_buy_orders["deleted_at"].isnull() | (new_buy_orders["deleted_at"] == "")]
    if len(new_buy_orders) == 0: new_buy_orders = None

if update_buy_orders is not None:
    update_buy_orders = update_buy_orders.merge(
        bo_snapshot[["remoteId", "optiply_id", "concat_attributes"]],
        on="remoteId", how="left", suffixes=("", "_snap"))

    delete_buy_orders = update_buy_orders[~update_buy_orders["deleted_at"].isnull() & (update_buy_orders["deleted_at"] != "")]
    delete_buy_orders = delete_buy_orders.drop(columns=["concat_attributes_snap"])

    update_buy_orders = update_buy_orders[update_buy_orders["deleted_at"].isnull() | (update_buy_orders["deleted_at"] == "")]
    update_buy_orders = update_buy_orders[update_buy_orders["concat_attributes"] != update_buy_orders["concat_attributes_snap"]]
    update_buy_orders = update_buy_orders.drop(columns=["concat_attributes_snap"])

    # Supplier change handling: if supplierId changed, delete + recreate
    if update_buy_orders is not None and len(update_buy_orders) > 0 and "supplierId" in bo_snapshot.columns:
        supplier_changed = []
        for idx, row in update_buy_orders.iterrows():
            match = bo_snapshot[bo_snapshot["remoteId"] == row["remoteId"]]
            if len(match) > 0:
                old_s = str(match.iloc[0].get("supplierId", ""))
                new_s = str(row.get("supplierId", ""))
                if old_s != new_s and old_s != "nan" and new_s != "nan":
                    supplier_changed.append(row["remoteId"])
        if supplier_changed:
            print(f"Supplier changed on {len(supplier_changed)} buy orders -- delete + recreate")
            changed_mask = update_buy_orders["remoteId"].isin(supplier_changed)
            changed_rows = update_buy_orders[changed_mask]
            changed_in_snap = bo_snapshot[bo_snapshot["remoteId"].isin(supplier_changed)]
            if delete_buy_orders is not None:
                delete_buy_orders = pd.concat([delete_buy_orders, changed_in_snap], ignore_index=True)
            else:
                delete_buy_orders = changed_in_snap
            if new_buy_orders is not None:
                new_buy_orders = pd.concat([new_buy_orders, changed_rows], ignore_index=True)
            else:
                new_buy_orders = changed_rows
            update_buy_orders = update_buy_orders[~changed_mask]

    if update_buy_orders is not None and len(update_buy_orders) == 0: update_buy_orders = None
    if delete_buy_orders is not None and len(delete_buy_orders) == 0: delete_buy_orders = None
else:
    delete_buy_orders = None

del bo_snapshot

In [ ]:
new_records = new_buy_orders
del new_buy_orders
update_records = update_buy_orders
del update_buy_orders
delete_records = delete_buy_orders
del delete_buy_orders

entity = "buyOrders"
snapshot_name = "buyOrders"

print(f"BuyOrders -- DELETE: {len(delete_records) if delete_records is not None else 0}, "
      f"POST: {len(new_records) if new_records is not None else 0}, "
      f"PATCH: {len(update_records) if update_records is not None else 0}")

In [ ]:
# DELETE
if delete_records is not None:
    delete_records["optiply_id"] = delete_records["optiply_id"].astype(float).astype(int)
    total_records = len(delete_records)
    delete_records = delete_records.reset_index(drop=True)
    try:
        for i, row in delete_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            response = delete_optiply(api_creds_short, auth, optiply_id, entity=entity)
            if response.status_code == 404:
                print(f"Record {optiply_id} is already deleted in Optiply, skipping.")
                continue
            if not response.ok:
                raise Exception(f"Failed to delete record {optiply_id} - Status: {response.status_code}")
            summary[entity]['deleted'] += 1
        delete_from_snapshot(delete_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    except Exception as e:
        raise Exception(f"ETL FAILED WHILE DELETING RECORDS\n{e}")
    del delete_records

# POST
if new_records is not None:
    new_records["remoteId"] = new_records["remoteId"].astype(str)
    new_records_ = new_records.copy().reset_index(drop=True)
    new_records_["optiply_id"] = None
    total_records = len(new_records_)
    try:
        for i, row in new_records_.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            payload = get_payload_function(row, entity)
            response = post_optiply(api_creds_short, auth, payload, entity=entity)
            new_records_.loc[i, "optiply_id"] = str(response.json()["data"]["id"])
            summary[entity]['created'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE POSTING RECORDS -- SNAPSHOTTING POSTED RECORDS")
    finally:
        new_records_ = new_records_[~new_records_["optiply_id"].isna()]
        new_records_["optiply_id"] = new_records_["optiply_id"].astype(float).astype(int)
        snapshot_records(new_records_, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del new_records
    del new_records_

# PATCH
if update_records is not None:
    update_records["remoteId"] = update_records["remoteId"].astype(str)
    update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
    update_records["response_code"] = None
    total_records = len(update_records)
    update_records = update_records.reset_index(drop=True)
    try:
        for i, row in update_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            payload = get_payload_function(row, entity)
            response = patch_optiply(api_creds_short, auth, payload, optiply_id, entity=entity)
            update_records.loc[i, "response_code"] = response.status_code
            summary[entity]['updated'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE PATCHING RECORDS -- SNAPSHOTTING PATCHED RECORDS")
    finally:
        update_records = update_records[update_records["response_code"] == 200]
        update_records = update_records.drop(columns=["response_code"])
        update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
        snapshot_records(update_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del update_records

gc.collect()

## Buy Order Lines
### Custom Mapping
Extend PurchaseOrder rows -> Optiply BuyOrderLines

Lines are extracted from the `rows` JSON array embedded in each PurchaseOrder record.
remoteId = `{purchaseNumber}-{position}`.

In [ ]:
# BuyOrderLines: Custom Mapping (Extend-specific)
buy_order_lines = None

if po_raw is not None and len(po_raw) > 0 and sync_buy_orders:
    products_snapshot = get_snapshot('products', SNAPSHOT_DIR)
    bo_snapshot = get_snapshot('buyOrders', SNAPSHOT_DIR)

    bol_records = []
    for idx, row in po_raw.iterrows():
        bo_remote_id = str(row.get('purchaseNumber', ''))
        if not bo_remote_id or bo_remote_id == 'nan':
            continue

        # Bidirectional guard
        ext_order = str(row.get('externalOrderNumber', ''))
        reference = str(row.get('reference', ''))
        if 'optiply' in ext_order.lower() or 'optiply' in reference.lower():
            continue

        # Skip cancelled
        status = str(row.get('status', '')).strip()
        if status == 'Cancelled':
            continue

        # Resolve buy order Optiply ID
        bo_optiply_id = None
        if bo_snapshot is not None:
            match = bo_snapshot[bo_snapshot['remoteId'] == bo_remote_id]
            if len(match) > 0 and pd.notna(match.iloc[0].get('optiply_id')):
                bo_optiply_id = match.iloc[0]['optiply_id']
        if bo_optiply_id is None:
            continue

        parsed_rows = parse_nested_json(row.get('rows', '[]'))
        for po_row in parsed_rows:
            if not isinstance(po_row, dict):
                continue
            product_number = str(po_row.get('productNumber', ''))
            if not product_number or product_number == 'nan':
                continue
            position = po_row.get('position', 0)
            subposition = po_row.get('subposition', 0)

            optiply_pid = None
            if products_snapshot is not None:
                match_p = products_snapshot[products_snapshot['remoteId'] == product_number]
                if len(match_p) > 0 and pd.notna(match_p.iloc[0].get('optiply_id')):
                    optiply_pid = match_p.iloc[0]['optiply_id']
            if optiply_pid is None:
                continue

            product_unit = po_row.get('purchaseDataProductUnit', {}) or {}
            qty = int(float(product_unit.get('quantity', 0) or 0))
            unit_price = float(product_unit.get('unitPrice', 0) or 0)
            subtotal = round(qty * unit_price, 2)

            expected_delivery = None
            edd = po_row.get('expectedDeliveryDate')
            if edd and pd.notna(edd):
                dt = handle_invalid_dates(edd)
                expected_delivery = None if pd.isna(dt) else str(dt)

            updated_at = expected_delivery
            if updated_at is None:
                scd = po_row.get('statusChangeDate')
                if scd and pd.notna(scd):
                    dt = handle_invalid_dates(scd)
                    updated_at = None if pd.isna(dt) else str(dt)

            bol_records.append({
                'remoteId': f"{bo_remote_id}-{position}-{subposition}",
                'buyOrderId': int(bo_optiply_id),
                'productId': int(optiply_pid),
                'quantity': qty,
                'subtotalValue': subtotal,
                'expectedDeliveryDate': expected_delivery,
                'updated_at': updated_at,
                'created_at': None,
                'deleted_at': None,
            })

    if bol_records:
        buy_order_lines = pd.DataFrame(bol_records)
        print(f"BuyOrderLines custom mapped: {len(buy_order_lines)}")
    else:
        print("No buy order lines extracted.")
else:
    print("No purchase orders data for buy order lines.")


### Global Mapping

In [ ]:
# BuyOrderLines: Global Mapping
print("Global Mapping BuyOrderLines")
if buy_order_lines is not None:
    buy_order_lines = clean_underscore_columns(buy_order_lines)
    buy_order_lines.columns = buy_order_lines.columns.str.lower()

    if "remoteid" in buy_order_lines.columns:
        buy_order_lines.rename(columns={"remoteid": "remoteId"}, inplace=True)
    buy_order_lines["remoteId"] = buy_order_lines["remoteId"].astype(str)

    if "buyorderid" in buy_order_lines.columns:
        buy_order_lines.rename(columns={"buyorderid": "buyOrderId"}, inplace=True)
    if "productid" in buy_order_lines.columns:
        buy_order_lines.rename(columns={"productid": "productId"}, inplace=True)
    if "subtotalvalue" in buy_order_lines.columns:
        buy_order_lines.rename(columns={"subtotalvalue": "subtotalValue"}, inplace=True)
        buy_order_lines["subtotalValue"] = round_numeric_to_2(buy_order_lines["subtotalValue"])

    if "quantity" in buy_order_lines.columns:
        buy_order_lines["quantity"] = round_to_0(buy_order_lines["quantity"])

    for date_col in ["expectedDeliveryDate", "created_at", "updated_at", "deleted_at"]:
        if date_col not in buy_order_lines.columns:
            buy_order_lines[date_col] = None
        elif buy_order_lines[date_col].dtype == 'object':
            buy_order_lines[date_col] = buy_order_lines[date_col].astype(str).str[:19] + "Z"

    buy_order_lines = buy_order_lines[buy_order_lines["remoteId"].notna()]
    buy_order_lines = buy_order_lines[buy_order_lines["remoteId"] != ""]
    buy_order_lines = buy_order_lines[buy_order_lines["remoteId"] != "nan"]
    buy_order_lines = buy_order_lines.drop_duplicates(subset=["remoteId"])

    concat_fields = buy_order_lines.columns[~buy_order_lines.columns.isin(
        ["expectedDeliveryDate", "created_at", "updated_at", "deleted_at", "buyOrderId", "productId"])]
    buy_order_lines["concat_attributes"] = concat_columns(buy_order_lines, concat_fields)

    if len(buy_order_lines) == 0:
        buy_order_lines = None
    else:
        print(f"BuyOrderLines after global mapping: {len(buy_order_lines)}")

In [ ]:
# BuyOrderLines: Snapshot comparison
bol_snapshot = get_snapshot("buyOrderLines", SNAPSHOT_DIR)
if bol_snapshot is not None and len(bol_snapshot) == 0:
    bol_snapshot = None

if bol_snapshot is not None and buy_order_lines is not None:
    bol_snapshot["remoteId"] = bol_snapshot["remoteId"].astype(str)
    buy_order_lines["remoteId"] = buy_order_lines["remoteId"].astype(str)

    new_bol = buy_order_lines[~buy_order_lines["remoteId"].isin(bol_snapshot["remoteId"])]
    update_bol = buy_order_lines[buy_order_lines["remoteId"].isin(bol_snapshot["remoteId"])]

    if len(new_bol) == 0: new_bol = None
    if len(update_bol) == 0: update_bol = None
else:
    new_bol = buy_order_lines
    update_bol = None

del buy_order_lines

if new_bol is not None:
    new_bol = new_bol[new_bol["deleted_at"].isnull() | (new_bol["deleted_at"] == "")]
    if len(new_bol) == 0: new_bol = None

if update_bol is not None:
    update_bol = update_bol.merge(
        bol_snapshot[["remoteId", "optiply_id", "concat_attributes"]],
        on="remoteId", how="left", suffixes=("", "_snap"))

    delete_bol = update_bol[~update_bol["deleted_at"].isnull() & (update_bol["deleted_at"] != "")]
    delete_bol = delete_bol.drop(columns=["concat_attributes_snap"])

    update_bol = update_bol[update_bol["deleted_at"].isnull() | (update_bol["deleted_at"] == "")]
    update_bol = update_bol[update_bol["concat_attributes"] != update_bol["concat_attributes_snap"]]
    update_bol = update_bol.drop(columns=["concat_attributes_snap"])

    if len(update_bol) == 0: update_bol = None
    if len(delete_bol) == 0: delete_bol = None
else:
    delete_bol = None

del bol_snapshot

In [ ]:
new_records = new_bol
del new_bol
update_records = update_bol
del update_bol
delete_records = delete_bol
del delete_bol

entity = "buyOrderLines"
snapshot_name = "buyOrderLines"

print(f"BuyOrderLines -- DELETE: {len(delete_records) if delete_records is not None else 0}, "
      f"POST: {len(new_records) if new_records is not None else 0}, "
      f"PATCH: {len(update_records) if update_records is not None else 0}")

In [ ]:
# DELETE
if delete_records is not None:
    delete_records["optiply_id"] = delete_records["optiply_id"].astype(float).astype(int)
    total_records = len(delete_records)
    delete_records = delete_records.reset_index(drop=True)
    try:
        for i, row in delete_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            response = delete_optiply(api_creds_short, auth, optiply_id, entity=entity)
            if response.status_code == 404:
                print(f"Record {optiply_id} is already deleted in Optiply, skipping.")
                continue
            if not response.ok:
                raise Exception(f"Failed to delete record {optiply_id} - Status: {response.status_code}")
            summary[entity]['deleted'] += 1
        delete_from_snapshot(delete_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    except Exception as e:
        raise Exception(f"ETL FAILED WHILE DELETING RECORDS\n{e}")
    del delete_records

# POST
if new_records is not None:
    new_records["remoteId"] = new_records["remoteId"].astype(str)
    new_records_ = new_records.copy().reset_index(drop=True)
    new_records_["optiply_id"] = None
    total_records = len(new_records_)
    try:
        for i, row in new_records_.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            payload = get_payload_function(row, entity)
            response = post_optiply(api_creds_short, auth, payload, entity=entity)
            new_records_.loc[i, "optiply_id"] = str(response.json()["data"]["id"])
            summary[entity]['created'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE POSTING RECORDS -- SNAPSHOTTING POSTED RECORDS")
    finally:
        new_records_ = new_records_[~new_records_["optiply_id"].isna()]
        new_records_["optiply_id"] = new_records_["optiply_id"].astype(float).astype(int)
        snapshot_records(new_records_, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del new_records
    del new_records_

# PATCH
if update_records is not None:
    update_records["remoteId"] = update_records["remoteId"].astype(str)
    update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
    update_records["response_code"] = None
    total_records = len(update_records)
    update_records = update_records.reset_index(drop=True)
    try:
        for i, row in update_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            payload = get_payload_function(row, entity)
            response = patch_optiply(api_creds_short, auth, payload, optiply_id, entity=entity)
            update_records.loc[i, "response_code"] = response.status_code
            summary[entity]['updated'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE PATCHING RECORDS -- SNAPSHOTTING PATCHED RECORDS")
    finally:
        update_records = update_records[update_records["response_code"] == 200]
        update_records = update_records.drop(columns=["response_code"])
        update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
        snapshot_records(update_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del update_records

gc.collect()

## Receipt Lines
### Custom Mapping
Extend PurchaseOrder shipment positions -> Optiply ReceiptLines

ReceiptLines track received quantities. Extracted from PurchaseOrder rows
that have received positions (via shipments or row-level received quantities).

In [ ]:
# ReceiptLines: Custom Mapping (Extend-specific)
receipt_lines = None

if po_raw is not None and len(po_raw) > 0 and sync_buy_orders:
    products_snapshot = get_snapshot('products', SNAPSHOT_DIR)
    bo_snapshot = get_snapshot('buyOrders', SNAPSHOT_DIR)
    bol_snapshot = get_snapshot('buyOrderLines', SNAPSHOT_DIR)

    rl_records = []
    for idx, row in po_raw.iterrows():
        bo_remote_id = str(row.get('purchaseNumber', ''))
        if not bo_remote_id or bo_remote_id == 'nan':
            continue

        # Bidirectional guard
        ext_order = str(row.get('externalOrderNumber', ''))
        reference = str(row.get('reference', ''))
        if 'optiply' in ext_order.lower() or 'optiply' in reference.lower():
            continue

        status = str(row.get('status', '')).strip()
        if status == 'Cancelled':
            continue

        # Only process orders that have received items
        is_received = row.get('isReceived', False)
        if str(is_received).lower() not in ('true', '1'):
            continue

        # Build shipment lookup: position -> shipmentNumber
        shipments = parse_nested_json(row.get('shipments', '[]'))
        shipment_lookup = {}
        for s in shipments:
            if isinstance(s, dict):
                s_positions = s.get('positions', []) or []
                s_number = s.get('shipmentNumber', '')
                for sp in s_positions:
                    if isinstance(sp, dict):
                        sp_pos = sp.get('position', 0)
                        sp_subpos = sp.get('subposition', 0)
                        key = (sp_pos, sp_subpos)
                        if key not in shipment_lookup and s_number:
                            shipment_lookup[key] = str(s_number)

        parsed_rows = parse_nested_json(row.get('rows', '[]'))
        for po_row in parsed_rows:
            if not isinstance(po_row, dict):
                continue

            # Only process rows with received status
            row_status = str(po_row.get('rowStatus', '')).strip().lower()
            if row_status != 'received':
                continue

            product_number = str(po_row.get('productNumber', ''))
            if not product_number or product_number == 'nan':
                continue
            position = po_row.get('position', 0)
            subposition = po_row.get('subposition', 0)
            bol_remote_id = f"{bo_remote_id}-{position}-{subposition}"

            # Quantity from purchaseDataProductUnit (product-level qty)
            product_unit = po_row.get('purchaseDataProductUnit', {}) or {}
            qty = int(float(product_unit.get('quantity', 0) or 0))
            if qty <= 0:
                continue

            # Resolve buyOrderLineId from snapshot
            bol_optiply_id = None
            if bol_snapshot is not None:
                match = bol_snapshot[bol_snapshot['remoteId'] == bol_remote_id]
                if len(match) > 0 and pd.notna(match.iloc[0].get('optiply_id')):
                    bol_optiply_id = match.iloc[0]['optiply_id']
            if bol_optiply_id is None:
                continue

            # remoteId: use shipmentNumber if available, else derive from BOL
            shipment_key = (position, subposition)
            if shipment_key in shipment_lookup:
                rl_remote_id = shipment_lookup[shipment_key]
            else:
                rl_remote_id = f"RL-{bol_remote_id}"

            # Occurred: use statusChangeDate or shippedDate
            occurred = None
            scd = po_row.get('statusChangeDate')
            if scd and pd.notna(scd):
                dt = handle_invalid_dates(scd)
                occurred = None if pd.isna(dt) else str(dt)
            if occurred is None:
                shipped = row.get('shippedDate')
                if shipped and pd.notna(shipped):
                    dt = handle_invalid_dates(shipped)
                    occurred = None if pd.isna(dt) else str(dt)
            if occurred is None:
                occurred = datetime.datetime.utcnow().isoformat()

            rl_records.append({
                'remoteId': rl_remote_id,
                'buyOrderLineId': int(bol_optiply_id),
                'quantity': qty,
                'occurred': occurred,
                'created_at': None,
                'updated_at': occurred,
                'deleted_at': None,
            })

    if rl_records:
        receipt_lines = pd.DataFrame(rl_records)
        print(f"ReceiptLines custom mapped: {len(receipt_lines)}")
    else:
        print("No receipt lines extracted (no received POs or no resolved FKs).")
else:
    print("No purchase orders data for receipt lines.")


### Global Mapping

In [ ]:
# ReceiptLines: Global Mapping
print("Global Mapping ReceiptLines")
if receipt_lines is not None:
    receipt_lines = clean_underscore_columns(receipt_lines)
    receipt_lines.columns = receipt_lines.columns.str.lower()

    if "remoteid" in receipt_lines.columns:
        receipt_lines.rename(columns={"remoteid": "remoteId"}, inplace=True)
    receipt_lines["remoteId"] = receipt_lines["remoteId"].astype(str)

    if "buyorderlineid" in receipt_lines.columns:
        receipt_lines.rename(columns={"buyorderlineid": "buyOrderLineId"}, inplace=True)

    if "quantity" in receipt_lines.columns:
        receipt_lines["quantity"] = round_to_0(receipt_lines["quantity"])
        receipt_lines = receipt_lines[receipt_lines["quantity"] > 0]

    for date_col in ["occurred", "created_at", "updated_at", "deleted_at"]:
        if date_col not in receipt_lines.columns:
            receipt_lines[date_col] = None
        elif receipt_lines[date_col].dtype == 'object':
            receipt_lines[date_col] = receipt_lines[date_col].astype(str).str[:19] + "Z"

    receipt_lines = receipt_lines[receipt_lines["remoteId"].notna()]
    receipt_lines = receipt_lines[receipt_lines["remoteId"] != ""]
    receipt_lines = receipt_lines[receipt_lines["remoteId"] != "nan"]
    receipt_lines = receipt_lines.drop_duplicates(subset=["remoteId"])

    concat_fields = receipt_lines.columns[~receipt_lines.columns.isin(
        ["created_at", "updated_at", "deleted_at", "buyOrderLineId"])]
    receipt_lines["concat_attributes"] = concat_columns(receipt_lines, concat_fields)

    if len(receipt_lines) == 0:
        receipt_lines = None
    else:
        print(f"ReceiptLines after global mapping: {len(receipt_lines)}")

In [ ]:
# ReceiptLines: Snapshot comparison
rl_snapshot = get_snapshot("receiptLines", SNAPSHOT_DIR)
if rl_snapshot is not None and len(rl_snapshot) == 0:
    rl_snapshot = None

if rl_snapshot is not None and receipt_lines is not None:
    rl_snapshot["remoteId"] = rl_snapshot["remoteId"].astype(str)
    receipt_lines["remoteId"] = receipt_lines["remoteId"].astype(str)

    new_rl = receipt_lines[~receipt_lines["remoteId"].isin(rl_snapshot["remoteId"])]
    update_rl = receipt_lines[receipt_lines["remoteId"].isin(rl_snapshot["remoteId"])]

    if len(new_rl) == 0: new_rl = None
    if len(update_rl) == 0: update_rl = None
else:
    new_rl = receipt_lines
    update_rl = None

del receipt_lines

if new_rl is not None:
    new_rl = new_rl[new_rl["deleted_at"].isnull() | (new_rl["deleted_at"] == "")]
    if len(new_rl) == 0: new_rl = None

if update_rl is not None:
    update_rl = update_rl.merge(
        rl_snapshot[["remoteId", "optiply_id", "concat_attributes"]],
        on="remoteId", how="left", suffixes=("", "_snap"))

    delete_rl = update_rl[~update_rl["deleted_at"].isnull() & (update_rl["deleted_at"] != "")]
    delete_rl = delete_rl.drop(columns=["concat_attributes_snap"])

    update_rl = update_rl[update_rl["deleted_at"].isnull() | (update_rl["deleted_at"] == "")]
    update_rl = update_rl[update_rl["concat_attributes"] != update_rl["concat_attributes_snap"]]
    update_rl = update_rl.drop(columns=["concat_attributes_snap"])

    if len(update_rl) == 0: update_rl = None
    if len(delete_rl) == 0: delete_rl = None
else:
    delete_rl = None

del rl_snapshot

In [ ]:
new_records = new_rl
del new_rl
update_records = update_rl
del update_rl
delete_records = delete_rl
del delete_rl

entity = "receiptLines"
snapshot_name = "receiptLines"

print(f"ReceiptLines -- DELETE: {len(delete_records) if delete_records is not None else 0}, "
      f"POST: {len(new_records) if new_records is not None else 0}, "
      f"PATCH: {len(update_records) if update_records is not None else 0}")

In [ ]:
# DELETE
if delete_records is not None:
    delete_records["optiply_id"] = delete_records["optiply_id"].astype(float).astype(int)
    total_records = len(delete_records)
    delete_records = delete_records.reset_index(drop=True)
    try:
        for i, row in delete_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            response = delete_optiply(api_creds_short, auth, optiply_id, entity=entity)
            if response.status_code == 404:
                print(f"Record {optiply_id} is already deleted in Optiply, skipping.")
                continue
            if not response.ok:
                raise Exception(f"Failed to delete record {optiply_id} - Status: {response.status_code}")
            summary[entity]['deleted'] += 1
        delete_from_snapshot(delete_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    except Exception as e:
        raise Exception(f"ETL FAILED WHILE DELETING RECORDS\n{e}")
    del delete_records

# POST
if new_records is not None:
    new_records["remoteId"] = new_records["remoteId"].astype(str)
    new_records_ = new_records.copy().reset_index(drop=True)
    new_records_["optiply_id"] = None
    total_records = len(new_records_)
    try:
        for i, row in new_records_.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            payload = get_payload_function(row, entity)
            response = post_optiply(api_creds_short, auth, payload, entity=entity)
            new_records_.loc[i, "optiply_id"] = str(response.json()["data"]["id"])
            summary[entity]['created'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE POSTING RECORDS -- SNAPSHOTTING POSTED RECORDS")
    finally:
        new_records_ = new_records_[~new_records_["optiply_id"].isna()]
        new_records_["optiply_id"] = new_records_["optiply_id"].astype(float).astype(int)
        snapshot_records(new_records_, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del new_records
    del new_records_

# PATCH
if update_records is not None:
    update_records["remoteId"] = update_records["remoteId"].astype(str)
    update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
    update_records["response_code"] = None
    total_records = len(update_records)
    update_records = update_records.reset_index(drop=True)
    try:
        for i, row in update_records.iterrows():
            print(f"Record: {i+1} of Total: {total_records}")
            optiply_id = int(row['optiply_id'])
            payload = get_payload_function(row, entity)
            response = patch_optiply(api_creds_short, auth, payload, optiply_id, entity=entity)
            update_records.loc[i, "response_code"] = response.status_code
            summary[entity]['updated'] += 1
    except:
        raise Exception(f"ETL FAILED WHILE PATCHING RECORDS -- SNAPSHOTTING PATCHED RECORDS")
    finally:
        update_records = update_records[update_records["response_code"] == 200]
        update_records = update_records.drop(columns=["response_code"])
        update_records["optiply_id"] = update_records["optiply_id"].astype(float).astype(int)
        snapshot_records(update_records, snapshot_name, SNAPSHOT_DIR, pk="remoteId")
    del update_records

gc.collect()

In [ ]:
print("=" * 50)
print("ETL Summary -- Extend Commerce")
print("=" * 50)
for entity_name, counts in summary.items():
    total = sum(counts.values())
    if total > 0:
        print(f"  {entity_name}: {counts}")
    else:
        print(f"  {entity_name}: no changes")
print("=" * 50)